# Lab 5.3: Points, Rules and a Confounded Shape

Run the cells in order. One cell, marked OPTIONAL, invites you to change a number. All randomness is seeded.

Part A uses the Breast Cancer Wisconsin (Diagnostic) data bundled with scikit-learn (no download). Part B uses synthetic patient records generated in the notebook. Part C uses German Credit from OpenML.

## Part A. A points card for a medical decision

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

bc = load_breast_cancer(as_frame=True)
Xb, yb = bc.data, (bc.target == 0).astype(int)        # 1 = malignant
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)
# yes/no items: is the measurement above its training median?
med = Xtr.median()
Btr, Bte = (Xtr > med).astype(int), (Xte > med).astype(int)
# L1 logistic regression picks a short list of items
for C in [0.01, 0.02, 0.03, 0.05, 0.1]:
    l1 = LogisticRegression(penalty="l1", solver="liblinear", C=C).fit(Btr, ytr)
    k = (l1.coef_[0] != 0).sum()
    if k >= 4: break
items = Btr.columns[l1.coef_[0] != 0]
print("C =", C, "items:", list(items))

C = 0.03 items: ['mean concave points', 'worst radius', 'worst area', 'worst concavity', 'worst concave points']


In [2]:
lr = LogisticRegression(max_iter=2000).fit(Btr[items], ytr)
w = lr.coef_[0]
points = np.round(w / np.abs(w).min()).astype(int)
card = pd.DataFrame({"item": [f"{c} above median" for c in items], "weight": w.round(2), "points": points})
print(card.to_string(index=False))
score_te = Bte[items].values @ points
print("\nTest AUC, exact weights:", round(roc_auc_score(yte, lr.predict_proba(Bte[items])[:, 1]), 3))
print("Test AUC, integer points:", round(roc_auc_score(yte, score_te), 3))
print("\nShare malignant by total points (test set):")
print(pd.Series(yte.values).groupby(score_te).agg(["mean", "size"]).round(2))

                             item  weight  points
 mean concave points above median    1.44       1
        worst radius above median    1.58       1
          worst area above median    1.67       2
     worst concavity above median    2.05       2
worst concave points above median    1.10       1

Test AUC, exact weights: 0.952
Test AUC, integer points: 0.954

Share malignant by total points (test set):
   mean  size
0  0.00    50
1  0.00     6
2  0.08    12
3  0.00    12
4  0.12    16
5  0.20     5
6  0.25     4
7  0.89    66


**Checkpoint A:** a card of four to six items with small integer points; the test AUC of the points is within about 0.01 of the exact weights, and the share of malignant cases rises overall with the total, from 0 at 0 points to about 0.9 at the maximum; small groups in the middle wobble. Write the card on paper: could a person check it by hand?

## Part B. An additive model learns a treatment as protection

Synthetic pneumonia-style records: asthma raises the true risk of death by 0.6 on the log-odds scale, but every asthmatic patient is sent to intensive care, which changes it by `ICU_EFFECT`.

In [3]:
ICU_EFFECT = -1.5          # OPTIONAL: try 0.0 or -0.5 and rerun Part B

from interpret.glassbox import ExplainableBoostingClassifier
r = np.random.default_rng(0); n = 20000
age = np.clip(18 + 82 * r.beta(2.2, 1.6, n), 18, 100)
asthma = (r.random(n) < 0.10).astype(float)
resp = np.clip(r.normal(22, 6, n), 8, 50)
logit = (-3.6 + 0.045 * (age - 60) + 0.6 * asthma + 0.09 * np.maximum(resp - 26, 0)
         + 0.06 * np.maximum(14 - resp, 0) + ICU_EFFECT * asthma)
yp = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
Xp = pd.DataFrame(dict(age=age, asthma=asthma, resp=resp))
Ptr, Pte, ptr, pte = train_test_split(Xp, yp, test_size=0.3, random_state=0, stratify=yp)
ebm = ExplainableBoostingClassifier(interactions=0, random_state=0).fit(Ptr, ptr)
i = ebm.term_names_.index("asthma")
print("asthma term (no asthma, asthma):", ebm.term_scores_[i][1:3].round(2))
print("learned asthma effect:", round(ebm.term_scores_[i][2] - ebm.term_scores_[i][1], 2), "| true untreated effect: +0.60")
print("test AUC:", round(roc_auc_score(pte, ebm.predict_proba(Pte)[:, 1]), 3))

asthma term (no asthma, asthma): [ 0.1  -0.84]
learned asthma effect: -0.94 | true untreated effect: +0.60
test AUC: 0.725


**Checkpoint B1:** with `ICU_EFFECT = -1.5`, the learned asthma effect is about -0.9: the model says asthma protects.

In [4]:
# Edit the term: give asthma its untreated effect (+0.6) relative to no asthma
edited = ebm.term_scores_[i].copy()
edited[2] = edited[1] + 0.6
ebm.term_scores_[i] = edited
print("edited asthma term:", ebm.term_scores_[i][1:3].round(2))
print("test AUC after edit:", round(roc_auc_score(pte, ebm.predict_proba(Pte)[:, 1]), 3))

edited asthma term: [0.1 0.7]
test AUC after edit: 0.702


**Checkpoint B2:** the test AUC falls after the edit (from about 0.73 to about 0.70). The edited model fits these data worse, because in these data asthmatic patients really did die less often. It is the right model only if it will be used to decide who needs intensive care. Write down who should approve such an edit and what the record must say.

## Part C. Credible intervals on German Credit

A Bayesian logistic regression with a Normal(0, 1) prior on each effect, using the Laplace approximation (the method of the Deep box in Chapter 5). Rare categories get wide intervals.

In [5]:
def load_credit():
    """German Credit (Statlog) from OpenML (credit-g); falls back to data/german_credit.csv if offline."""
    try:
        from sklearn.datasets import fetch_openml
        df = fetch_openml("credit-g", version=1, as_frame=True).frame.copy()
    except Exception:
        df = pd.read_csv("../../../data/german_credit.csv")
    df["bad"] = (df["class"] == "bad").astype(int)
    return df.drop(columns=["class"])

d = load_credit()
yc = d.pop("bad").values.astype(float)
F = pd.get_dummies(d[["checking_status", "purpose"]].astype(str), drop_first=True, dtype=float)
A = np.column_stack([np.ones(len(F)), F.values])
prior_var = np.r_[10.0, np.ones(F.shape[1])]
wt = np.zeros(A.shape[1])
for _ in range(50):
    p = 1 / (1 + np.exp(-A @ wt))
    H = (A.T * (p * (1 - p))) @ A + np.diag(1 / prior_var)
    wt += np.linalg.solve(H, A.T @ (yc - p) - wt / prior_var)
sd = np.sqrt(np.diag(np.linalg.inv(H)))
out = pd.DataFrame({"n": F.sum().astype(int).values, "odds ratio": np.exp(wt[1:]),
                    "2.5%": np.exp(wt[1:] - 1.96 * sd[1:]), "97.5%": np.exp(wt[1:] + 1.96 * sd[1:])}, index=F.columns)
out.round(2).sort_values("n")

,n,odds ratio,2.5%,97.5%
purpose_retraining,9,0.46,0.12,1.71
purpose_domestic appliance,12,0.88,0.29,2.66
purpose_other,12,0.92,0.32,2.61
purpose_repairs,22,1.05,0.43,2.56
purpose_education,50,1.62,0.82,3.18
checking_status_>=200,63,0.48,0.26,0.89
purpose_used car,103,0.43,0.23,0.80
purpose_furniture/equipment,181,0.78,0.47,1.28
purpose_new car,234,1.11,0.69,1.77
checking_status_<0,274,1.56,1.11,2.20


**Checkpoint C:** categories with few applicants (for example the rarest `purpose` values) have intervals that span both sides of 1, sometimes very widely; common categories such as `checking_status_no checking` have narrow intervals. Which of these effects would you be willing to explain to an applicant?